## Objective

To fine-tune the pretrained GPT-2 model on a custom text style and compare its output with the original GPT-2 model using the same prompt.

Install Required Libraries




In [1]:
!pip install -q transformers datasets accelerate torch

Import Libraries

In [2]:
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
import torch

Load the GPT-2 Model and Tokenizer

In [3]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

# GPT-2 does not have a padding token by default
tokenizer.pad_token = tokenizer.eos_token

print("GPT-2 model loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 model loaded successfully!


Create the Custom Dataset

In [4]:
texts = [
    "The young scientist lived in a small town surrounded by quiet green hills.",
    "Every morning, he walked to his laboratory before the sun rose.",
    "His laboratory was an old building that had once been a railway station.",
    "The building had large windows and a wooden door covered with scratches.",
    "Inside the laboratory, hundreds of books were arranged on tall shelves.",
    "The scientist spent most of his time studying unusual scientific discoveries.",
    "He was especially interested in objects that had no known explanation.",
    "One rainy evening, he received a mysterious package.",
    "The package had no name or address written on it.",
    "It was wrapped in a thick brown cloth and tied with a silver thread.",
    "The scientist carefully placed the package on his wooden table.",
    "He looked at it for several minutes before touching it.",
    "There was something unusual about the package.",
    "A faint blue light was coming from underneath the cloth.",
    "He switched off the laboratory lights and watched the glow become brighter.",
    "The strange light moved slowly across the surface of the package.",
    "The scientist became curious and carefully removed the silver thread.",
    "Under the cloth, he found a small metal box.",
    "The box was covered with strange symbols.",
    "None of the symbols looked familiar to him.",
    "He picked up the box and felt a gentle vibration in his hands.",
    "The vibration stopped as soon as he placed the box on the table.",
    "He examined the box using a small magnifying glass.",
    "There was a tiny circular mark on the top of the box.",
    "The mark looked like a star surrounded by three rings.",
    "The scientist opened his notebook and drew the symbol carefully.",
    "He searched through his old books for the same symbol.",
    "After several hours, he found something interesting.",
    "One of the oldest books in the laboratory contained a similar drawing.",
    "The drawing was almost five hundred years old.",
    "According to the book, the symbol represented a hidden source of energy.",
    "The scientist could hardly believe what he was reading.",
    "He turned the pages slowly and discovered a handwritten message.",
    "The message said that the box should never be opened after midnight.",
    "The scientist looked at the clock on the wall.",
    "It was already eleven forty-five.",
    "He decided to wait until morning.",
    "He covered the box and locked the laboratory door.",
    "Then he went home and tried to sleep.",
    "However, he could not stop thinking about the mysterious box.",
    "At exactly midnight, a bright light appeared outside his window.",
    "The scientist quickly got out of bed and looked outside.",
    "A strange blue light was floating above the old laboratory.",
    "He ran back to the building as quickly as he could.",
    "When he entered the laboratory, the box was glowing.",
    "The cover of the box had opened by itself.",
    "Inside the box, he found a small glass sphere.",
    "The sphere contained a moving cloud of silver light.",
    "The scientist slowly reached toward the sphere.",
    "As his fingers touched the glass, the laboratory became completely silent.",
    "Every clock in the room stopped at the same time.",
    "The scientist pulled his hand away in surprise.",
    "The sphere began to float above the table.",
    "A thin beam of light appeared from its center.",
    "The beam formed a bright circle on the laboratory floor.",
    "Inside the circle, the scientist saw an image of a distant mountain.",
    "The mountain was surrounded by a thick blue mist.",
    "He recognized the place from an old map.",
    "It was a mountain located far beyond the northern forest.",
    "The scientist wondered why the sphere was showing him that place.",
    "He opened his notebook and wrote down everything he had seen.",
    "Suddenly, the sphere dropped back into the box.",
    "The light disappeared, and the laboratory became dark again.",
    "The scientist stood silently beside the table.",
    "He knew that the mystery had only just begun.",
    "The next morning, he packed his equipment for a journey.",
    "He placed the mysterious box inside a strong metal case.",
    "He also carried his notebook, compass, flashlight, and camera.",
    "Before leaving, he locked the laboratory carefully.",
    "He told no one about the strange discovery.",
    "The journey to the northern mountain would take several days.",
    "The scientist started walking before sunrise.",
    "The road passed through a quiet forest filled with tall trees.",
    "Birds flew between the branches as the morning light appeared.",
    "After several hours, the scientist reached a narrow wooden bridge.",
    "A fast river flowed beneath the bridge.",
    "He crossed carefully while holding his equipment tightly.",
    "On the other side, he noticed something strange.",
    "There were fresh footprints in the wet ground.",
    "The footprints looked as if someone had followed him.",
    "The scientist looked behind him, but nobody was there.",
    "He continued walking through the forest.",
    "The footprints appeared again several meters ahead.",
    "He became nervous but decided not to stop.",
    "The forest became darker as he walked deeper inside.",
    "A cold wind moved through the trees.",
    "The scientist heard a soft whisper behind him.",
    "He turned around quickly.",
    "There was only an empty path behind him.",
    "He took a deep breath and continued forward.",
    "By evening, he reached a small abandoned cabin.",
    "The cabin had a broken window and an old wooden door.",
    "He decided to spend the night there.",
    "Inside the cabin, he found a fireplace and a dusty chair.",
    "There was also a map hanging on the wall.",
    "The map showed the northern mountain.",
    "A red circle had been drawn around its highest peak.",
    "The scientist carefully removed the map from the wall.",
    "He noticed a hidden message written behind it.",
    "The message warned travelers not to enter the mountain alone.",
    "The scientist wondered who had written the warning.",
    "He placed the map inside his notebook.",
    "That night, he heard footsteps outside the cabin.",
    "The footsteps moved slowly around the building.",
    "He turned off his flashlight and remained completely silent.",
    "After several minutes, the footsteps stopped.",
    "The scientist looked through the broken window.",
    "He saw a shadow standing between the trees.",
    "The shadow disappeared before he could identify it.",
    "He stayed awake until sunrise.",
    "In the morning, the forest was completely quiet.",
    "The scientist left the cabin and continued toward the mountain.",
    "The path became steeper as he climbed.",
    "Large rocks covered the ground.",
    "Snow began to fall around him.",
    "The air became colder with every step.",
    "After several hours, he reached the base of the mountain.",
    "A large stone entrance stood between two cliffs.",
    "The entrance was covered with the same symbols as the mysterious box.",
    "The scientist knew he had found the place from the vision.",
    "He opened his metal case and removed the box.",
    "The box began to glow as soon as he brought it near the entrance.",
    "A deep sound came from inside the mountain.",
    "The stone entrance slowly moved aside.",
    "A dark tunnel appeared behind it.",
    "The scientist turned on his flashlight and entered the tunnel.",
    "The walls were covered with ancient drawings.",
    "Some drawings showed stars and strange machines.",
    "Others showed people standing around a glowing sphere.",
    "The scientist photographed every drawing.",
    "The tunnel became narrower as he continued.",
    "Soon, he reached a large underground room.",
    "In the center of the room stood a stone table.",
    "A circular space was carved into the center of the table.",
    "The mysterious box fit perfectly inside the space.",
    "The scientist placed the box on the table.",
    "A bright light filled the room.",
    "The ancient drawings began to glow.",
    "A deep voice echoed through the chamber.",
    "The voice told him that the box had been waiting for him.",
    "The scientist looked around but could not see anyone.",
    "He asked who was speaking.",
    "The voice answered that the mountain protected an ancient invention.",
    "The invention had been created to store energy from the stars.",
    "The scientist listened carefully to every word.",
    "He asked why the box had been sent to him.",
    "The voice explained that the scientist had discovered the symbol years ago.",
    "He remembered studying the symbol during his university days.",
    "At that time, he had thought it was only a historical drawing.",
    "Now he understood that it was something much more important.",
    "A hidden door opened on the other side of the chamber.",
    "The scientist walked toward it slowly.",
    "Beyond the door was a room filled with strange machines.",
    "Thousands of small lights covered the walls.",
    "A large crystal stood at the center of the room.",
    "The crystal was brighter than anything the scientist had ever seen.",
    "He moved closer to examine it.",
    "The crystal showed images of cities, oceans, and distant planets.",
    "The scientist realized that the machine could store memories.",
    "He wondered who had built such an advanced device.",
    "The voice told him that the creators had disappeared long ago.",
    "Their knowledge had been hidden inside the mountain.",
    "The scientist felt both excited and afraid.",
    "He knew that such knowledge could change the world.",
    "But he also understood that it could be dangerous.",
    "The voice asked him to make an important decision.",
    "He could take the knowledge outside the mountain.",
    "Or he could leave everything hidden and protect the secret.",
    "The scientist thought about his work and his responsibility.",
    "He remembered why he had become a scientist.",
    "He wanted to discover new things and help people.",
    "He did not want his discovery to become a weapon.",
    "After thinking carefully, he chose to protect the secret.",
    "The lights inside the chamber slowly became dim.",
    "The voice thanked him for making the right decision.",
    "The crystal began to disappear into the stone floor.",
    "The machines stopped working one by one.",
    "The mysterious box returned to its normal appearance.",
    "The scientist picked it up carefully.",
    "He walked back toward the entrance of the mountain.",
    "The ancient symbols on the walls stopped glowing.",
    "The stone door opened once again.",
    "Cold sunlight entered the tunnel.",
    "The scientist stepped outside and looked at the sky.",
    "Snow was falling gently around the mountain.",
    "He felt a strange sense of peace.",
    "He knew that nobody would believe his story.",
    "Still, he wrote everything in his notebook.",
    "He decided to keep the notebook hidden in his laboratory.",
    "The journey home was long and quiet.",
    "He crossed the forest without seeing the mysterious shadow again.",
    "When he reached the old cabin, he found something unexpected.",
    "A small silver key was lying on the wooden floor.",
    "He picked it up and examined it carefully.",
    "The key had the same star symbol as the box.",
    "The scientist wondered whether the mystery was truly over.",
    "He placed the key inside his pocket.",
    "Then he continued his journey home.",
    "After two days, he finally reached his town.",
    "His laboratory looked exactly as he had left it.",
    "He unlocked the door and entered quietly.",
    "Everything was covered with a thin layer of dust.",
    "He placed the mysterious box inside a locked cabinet.",
    "Then he opened his notebook and began writing.",
    "He described the mountain, the underground chamber, and the crystal.",
    "He also described the voice and the ancient machines.",
    "He wrote for several hours without stopping.",
    "When he finished, he placed the notebook inside a hidden drawer.",
    "He locked the drawer with the silver key.",
    "For the first time in several days, he felt safe.",
    "The scientist returned to his normal research.",
    "Weeks passed without anything unusual happening.",
    "He almost began to believe that the mystery was finished.",
    "One evening, he noticed a faint blue light under the laboratory door.",
    "He slowly walked toward the light.",
    "The locked cabinet was glowing again.",
    "The scientist opened the cabinet with shaking hands.",
    "The mysterious box was shining brightly.",
    "He carefully removed it from the cabinet.",
    "The silver key inside his drawer began to vibrate.",
    "He realized that the key was connected to the box.",
    "He placed the key beside the box.",
    "A small hidden compartment opened on the side.",
    "Inside the compartment was a folded piece of paper.",
    "The scientist opened the paper carefully.",
    "There was only one sentence written on it.",
    "The sentence said that the first secret had been protected.",
    "It also said that the second secret was waiting.",
    "The scientist stared at the message in silence.",
    "He understood that his adventure was not finished.",
    "Outside the laboratory, the night sky was filled with stars.",
    "One star suddenly became brighter than the others.",
    "The scientist looked through the window.",
    "A narrow beam of blue light moved across the sky.",
    "It disappeared behind the distant hills.",
    "The scientist opened his notebook again.",
    "He wrote down the date and the time.",
    "Then he drew the star symbol on a new page.",
    "He decided that he would follow the light.",
    "The next morning, he prepared for another journey.",
    "This time, he knew that he had to be more careful.",
    "He packed his equipment and placed the mysterious box in his bag.",
    "Before leaving, he looked around his laboratory.",
    "He wondered how many secrets were still hidden in the world.",
    "Then he closed the door and began walking toward the hills.",
    "The morning sun slowly rose behind the mountains.",
    "A cool wind moved through the empty streets.",
    "The scientist followed the direction of the strange blue light.",
    "He did not know what he would discover next.",
    "But he knew that every discovery began with a single question.",
    "And he was ready to find the answer."
]

print("Number of lines:", len(texts))

Number of lines: 251


Create the Dataset

In [5]:
dataset = Dataset.from_dict({
    "text": texts
})

print("Number of lines:", len(texts))

Number of lines: 251


Tokenize the Dataset

In [6]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print("Dataset tokenized successfully!")

Map:   0%|          | 0/251 [00:00<?, ? examples/s]

Dataset tokenized successfully!


Prepare the Data Collator

In [7]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator prepared successfully!")

Data collator prepared successfully!


Set Training Parameters

In [9]:
training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=5e-5,
    report_to="none"
)

print("Training arguments created successfully!")

Training arguments created successfully!


Create the Trainer

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

print("Trainer created successfully!")

Trainer created successfully!


Fine-Tune GPT-2

In [11]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,3.322165
100,2.845527
150,2.373175
200,2.034735
250,1.972117
300,1.561697
350,1.666097


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=378, training_loss=2.1958640466922175, metrics={'train_runtime': 51.2835, 'train_samples_per_second': 14.683, 'train_steps_per_second': 7.371, 'total_flos': 49188225024000.0, 'train_loss': 2.1958640466922175, 'epoch': 3.0})

Save the Fine-Tuned Model

In [12]:
trainer.save_model("./gpt2-finetuned")
tokenizer.save_pretrained("./gpt2-finetuned")

print("Fine-tuned GPT-2 model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned GPT-2 model saved successfully!


Load the Original GPT-2 Model

In [13]:
original_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
original_tokenizer.pad_token = original_tokenizer.eos_token

original_model = GPT2LMHeadModel.from_pretrained("gpt2")

print("Original GPT-2 model loaded successfully!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Original GPT-2 model loaded successfully!


Load the Fine-Tuned GPT-2 Model

In [14]:
finetuned_tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-finetuned")
finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

finetuned_model = GPT2LMHeadModel.from_pretrained("./gpt2-finetuned")

print("Fine-tuned GPT-2 model loaded successfully!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Fine-tuned GPT-2 model loaded successfully!


 Define the Same Prompt from Assignment 2


In [15]:
prompt = "The young scientist opened the mysterious box and"

print("Prompt:", prompt)

Prompt: The young scientist opened the mysterious box and


Generate Output from Original GPT-2

In [16]:
def generate_text(model, tokenizer, prompt):
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_length=80,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            repetition_penalty=1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

original_output = generate_text(
    original_model,
    original_tokenizer,
    prompt
)

print("Original GPT-2 Output:")
print(original_output)

Original GPT-2 Output:
The young scientist opened the mysterious box and found a piece of paper.

"I had this paper," he said. "It was a piece of paper. I just took it and wrote it on it."

The scientist found a note from a man named Jack. The note read: "The man who made this paper is dead. He has died. He has died."




Generate Output from Fine-Tuned GPT-2

In [17]:
finetuned_output = generate_text(
    finetuned_model,
    finetuned_tokenizer,
    prompt
)

print("Fine-Tuned GPT-2 Output:")
print(finetuned_output)

Fine-Tuned GPT-2 Output:
The young scientist opened the mysterious box and carefully removed the silver key.

Inside, he found a mysterious box. Inside, he found a mysterious box.

He opened the box again.

The mysterious box opened again.

He opened the box again.

He opened the box again.

He opened the box again.

He opened the box again.


Before-and-After Comparison

In [18]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Original GPT-2",
        "Fine-Tuned GPT-2"
    ],
    "Output": [
        original_output,
        finetuned_output
    ]
})

comparison

,Model,Output
0,Original GPT-2,The young scientist opened the mysterious box ...
1,Fine-Tuned GPT-2,The young scientist opened the mysterious box ...


Analysis

The original GPT-2 generated general-purpose text, while the fine-tuned GPT-2 followed the writing style of the custom dataset more closely.
Two major differences:
1. Style: The fine-tuned model produced a more consistent mystery and storytelling style.
2. Vocabulary: The fine-tuned model used words and sentence patterns similar to the custom training data.

Conclusion

Fine-tuning successfully adapted GPT-2 to the custom writing style. The fine-tuned model generated more consistent and style-specific text compared with the original GPT-2 model.